In [14]:
import numpy             as     np
import pandas            as     pd
import geopandas         as     gpd
import os
import time
from   osgeo             import gdal
from   scipy.spatial     import cKDTree
from   joblib            import dump, load

In [15]:
# Bicubic Interpolation Functions
import numpy as np

def cubic_kernel(t):
    """
    Cubic interpolation kernel.
    """
    t = np.abs(t)
    a = -0.5  # Standard Catmull-Rom spline

    return np.where(
        t <= 1,
        (a + 2) * t**3 - (a + 3) * t**2 + 1,
        np.where(
            t < 2,
            a * t**3 - 5 * a * t**2 + 8 * a * t - 4 * a,
            0
        )
    )

def bicubic_interpolation(image, x, y, No_Data, Counter):
    """
    Bicubic interpolation that handles NaNs in the image.
    
    Parameters:
    image (numpy array): 2D grayscale image.
    x (float): x-coordinate of the target point.
    y (float): y-coordinate of the target point.
    
    Returns:
    float: Interpolated pixel value at (x, y), handling NaNs.
    """
    height, width = image.shape
    # Handle border cases using reflection padding
    if x < 1 or x > width - 3 or y < 1 or y > height - 3:
        image = np.pad(image, 2, mode='reflect')  # Mirror padding
        x += 2
        y += 2

    # Floor values to find surrounding grid points
    x0 = int(np.floor(x))
    y0 = int(np.floor(y))
    
    # Compute distances from the top-left pixel
    dx = x - x0
    dy = y - y0

    # Extract 4x4 neighborhood
    neighborhood = image[y0-1:y0+3, x0-1:x0+3]
    # Mask out NaN values
    valid_mask = ~np.isin(neighborhood, No_Data)
    if valid_mask.sum() < 8:
        return No_Data, Counter
    if valid_mask.sum() < 16:
        Counter = Counter + 1

    # Compute weights using the cubic kernel
    wx = cubic_kernel(np.arange(-1, 3) - dx)  # Weights in x direction
    wy = cubic_kernel(np.arange(-1, 3) - dy)  # Weights in y direction

    # Set NaN positions to 0 in neighborhood and weights
    #neighborhood = np.nan_to_num(neighborhood)
    neighborhood[neighborhood == No_Data] = np.mean(neighborhood[neighborhood != No_Data])

    wx *= valid_mask.sum(axis=0)  # Zero-out weights for columns with NaNs
    wy *= valid_mask.sum(axis=1)  # Zero-out weights for rows with NaNs
    # Normalize weights to ensure they sum to 1
    if np.sum(wx) > 0:
        wx /= np.sum(wx)
    if np.sum(wy) > 0:
        wy /= np.sum(wy)

    # Perform interpolation: first along x, then along y
    intermediate = np.dot(neighborhood, wx)  # Apply x weights
    result = np.dot(intermediate, wy)  # Apply y weights
    # Verify if the values is within the valid values range
    if not(np.min(neighborhood[neighborhood != No_Data]) <= result <= np.max(neighborhood[neighborhood != No_Data])):
        return No_Data, Counter
    # Update the number corrected interpolated windows
    if valid_mask.sum() < 16:
        Counter = Counter + 1
    #return np.clip(result,0, None)  # Ensure valid pixel range
    return result, Counter

# Bilinear Interpolation Functions
def bilinear_interpolation(image, x, y, No_Data, Counter):
    """
    Perform bilinear interpolation for a given point (x, y) in a 2D grayscale image,
    handling NaN values by adjusting interpolation weights.

    Parameters:
    image (numpy array): 2D grayscale image (may contain NaNs).
    x (float): x-coordinate of the target point.
    y (float): y-coordinate of the target point.

    Returns:
    float: Interpolated pixel value at (x, y).
    """
    height, width = image.shape

    # Get integer grid points (top-left corner of interpolation square)
    x0, y0 = int(np.floor(x)), int(np.floor(y))
    x1, y1 = min(x0 + 1, width - 1), min(y0 + 1, height - 1)  # Prevent out-of-bounds

    # Extract 2×2 neighborhood and corresponding coordinates
    neighborhood = np.array([
        [image[y0, x0], image[y0, x1]],  # Top row (y0)
        [image[y1, x0], image[y1, x1]]   # Bottom row (y1)
    ], dtype=np.float32)

    # Create mask for valid (non-NaN) values
    valid_mask = ~np.isin(neighborhood, No_Data)
    if valid_mask.sum() < 2:
        return No_Data, Counter

    # Replace NaNs with 0 for weighted sum calculations
    #neighborhood[~valid_mask] = 0
    neighborhood[~valid_mask] = np.mean(neighborhood[neighborhood != No_Data])

    # Compute interpolation weights
    wx = x - x0  # Distance from x0
    wy = y - y0  # Distance from y0

    # Interpolation along x-axis (handling NaNs)
    weights_x = np.array([1 - wx, wx])  # Linear weights in x-direction
    sum_weights_x = np.dot(weights_x, valid_mask)  # Normalize only valid values

    fx_y = np.dot(weights_x, neighborhood)  # Compute weighted sum
    fx_y[sum_weights_x > 0] /= sum_weights_x[sum_weights_x > 0]  # Normalize valid rows
    fx_y[sum_weights_x == 0] = np.nan  # If both x-values are NaN, set to NaN

    # Interpolation along y-axis (handling NaNs)
    weights_y = np.array([1 - wy, wy])
    valid_y = ~np.isnan(fx_y)  # Check valid intermediate values

    if not valid_y.any():
        return No_Data, Counter  # If both interpolated y-values are NaN, return NaN

    f_xy = np.dot(weights_y, np.nan_to_num(fx_y))  # Weighted sum
    f_xy /= np.dot(weights_y, valid_y)  # Normalize weights to valid values

    # Verify if the values is within the valid values range
    if not(np.min(neighborhood[neighborhood != No_Data]) <= f_xy <= np.max(neighborhood[neighborhood != No_Data])):
        return No_Data, Counter
    # Update the number corrected interpolated windows
    if valid_mask.sum() < 4:
        Counter = Counter + 1
    #return np.clip(f_xy, 0, None)  # Ensure valid pixel range
    return f_xy, Counter

In [ ]:
BIOMASS_POOL = 'HEIGHT'
# Load Files
Root_Directory = 'D:\\Tese_data_refinada\\DATA'
#Root_Directory = 'E:/Data/4_Data/_Test/'
# Get Files
Data_Paths  = dict()
Study_Areas = []
for Directory in os.listdir(Root_Directory):
    if os.path.isdir(Root_Directory + Directory) :
        Area             = Directory.split('_')[0]
        Data_Paths[Area] = []
        Study_Areas.append(Area)
        for File in os.listdir(Root_Directory + Directory + '/'):
            if File.endswith('.gpkg'):
                Data_Paths[Area].append(Root_Directory + Directory + '/' + File)
            elif File.endswith('.tif'):
                Data_Paths[Area].append(Root_Directory + Directory + '/' + File)
            # File_Source = File.split('_')[2]
            # if File_Source == 'CHM':
            #     continue
            # elif File_Source == 'FIELD':
            #     # Verify if is the Biomass Pool being mapped
            #     if File.split('_')[3] == BIOMASS_POOL:
            #         Data_Paths[Area].append(Root_Directory + Directory + '/' + File)
            #     else:
            #         continue
            # else:
            #     if File.endswith('.tif'):
            #         Data_Paths[Area].append(Root_Directory + Directory + '/' + File)


LISTA COMPLETA DE ENDEREÇOS CAPTADOS:

📍 ÁREA: Mafra
  1. D:\Tese_data_refinada\DATA\Mafra_2020\Mafra_20200000_ALOS2_BETAHH_A.tif
  2. D:\Tese_data_refinada\DATA\Mafra_2020\Mafra_20200000_ALOS2_BETAHV_A.tif
  3. D:\Tese_data_refinada\DATA\Mafra_2020\Mafra_20200000_ALOS2_BETAND_A.tif
  4. D:\Tese_data_refinada\DATA\Mafra_2020\Mafra_20200000_ALOS2_BETARATIO_A.tif
  5. D:\Tese_data_refinada\DATA\Mafra_2020\Mafra_20200000_ALOS2_GAMMAHH_A.tif
  6. D:\Tese_data_refinada\DATA\Mafra_2020\Mafra_20200000_ALOS2_GAMMAHV_A.tif
  7. D:\Tese_data_refinada\DATA\Mafra_2020\Mafra_20200000_ALOS2_GAMMAND_A.tif
  8. D:\Tese_data_refinada\DATA\Mafra_2020\Mafra_20200000_ALOS2_GAMMARATIO_A.tif
  9. D:\Tese_data_refinada\DATA\Mafra_2020\Mafra_20200000_ALOS2_GLCMBETAHHASM_A.tif
  10. D:\Tese_data_refinada\DATA\Mafra_2020\Mafra_20200000_ALOS2_GLCMBETAHHCONTRAST_A.tif
  11. D:\Tese_data_refinada\DATA\Mafra_2020\Mafra_20200000_ALOS2_GLCMBETAHHCORRELATION_A.tif
  12. D:\Tese_data_refinada\DATA\Mafra_2020\Mafra_202

In [21]:
# --- DEBUG: Listagem de todos os endereços captados ---
print("\n" + "="*50)
print("LISTA COMPLETA DE ENDEREÇOS CAPTADOS:")
print("="*50)

# Percorre o dicionário: Area é a chave (ex: 'Mafra'), Paths é a lista de ficheiros
for Area, Paths in Data_Paths.items():
    print(f"\n📍 ÁREA: {Area}")
    
    if not Paths:
        print("  ⚠️ Nenhum ficheiro encontrado para esta área.")
    else:
        for i, path in enumerate(Paths, 1):
            # i é o contador, path é o endereço completo do ficheiro
            print(f"  {i}. {path}")

print("\n" + "="*50)
print(f"TOTAL DE ÁREAS: {len(Data_Paths)}")
print("="*50)


LISTA COMPLETA DE ENDEREÇOS CAPTADOS:

TOTAL DE ÁREAS: 0


In [7]:
# Count time
Time_Process_Start = time.perf_counter()
# Create Datasets - Using Bicubig and NN interpolation
#Study_Areas        = ['SerraLousa']
Biomass_Pools      = [BIOMASS_POOL]
GeoDataFrames_Dict = dict()
Dataset            = ''
for Area in Study_Areas:
    print(Area)
    # Get the GeoPackage file and load it to a GeoPandas Dataframe
    GeoDataFrames_Dict[Area] = gpd.read_file([array for array in Data_Paths[Area] if array.endswith('.gpkg')][0])
    # Extract point coordinates from GeoPackage points
    GPKG_Coordinates_x = GeoDataFrames_Dict[Area].geometry.x.values
    GPKG_Coordinates_y = GeoDataFrames_Dict[Area].geometry.y.values  
    # Get Raster files paths
    Raster_Paths = [array for array in Data_Paths[Area] if array.endswith('.tif')]
    # Iterate Raster Paths to Apply Bicubic Interpolation
    for Path in Raster_Paths:
        T0               = time.perf_counter()
        # Get Dataset Name
        Prev_Dataset = Dataset
        Dataset      = Path.split('/')[-1].split('_')[2]
        # Load the raster file using GDAL
        Temporary_Raster = gdal.Open(Path)
        Temporary_Band   = Temporary_Raster.GetRasterBand(1)
        Temporary_Data   = Temporary_Band.ReadAsArray()
        Temporary_NoData = Temporary_Band.GetNoDataValue()
        # Get raster geotransform to convert pixel indices to coordinates and respective inverse
        Temporary_GT     = Temporary_Raster.GetGeoTransform()
        Temporary_INV_GT = gdal.InvGeoTransform(Temporary_GT)
        # Compute processing time
        T1                = time.perf_counter()
        Processing_Time   = T1 - T0
        # Convert to h:m:s
        hours, rem       = divmod(Processing_Time, 3600)
        minutes, seconds = divmod(rem, 60)
        print(f"Read Task\t\t: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}") 
        
        # Verify if it is a new dataset and if it is required to compute the nearest indices
        if Dataset != Prev_Dataset:
            #print('Computing NN...')
            # Perform Nearest Neighbour Interpolation (Centroid)
            # Get Columns and Rows
            Temporary_Columns, Temporary_Rows = np.meshgrid(np.arange(Temporary_Raster.RasterXSize), np.arange(Temporary_Raster.RasterYSize), indexing='xy')
            # Convert pixel indices to real-world coordinates
            X_Coordinates = Temporary_GT[0] + Temporary_Columns * Temporary_GT[1] + Temporary_Rows * Temporary_GT[2]
            Y_Coordinates = Temporary_GT[3] + Temporary_Columns * Temporary_GT[4] + Temporary_Rows * Temporary_GT[5]
            # Calculate the Centroid Coordinates
            X_Coordinates = X_Coordinates + Temporary_GT[1] / 2
            Y_Coordinates = Y_Coordinates + Temporary_GT[5] / 2
            # Flatten the Coordinate Arrays
            Temporary_Coordinates = np.column_stack((X_Coordinates.flatten(), Y_Coordinates.flatten()))
            Temporary_Values      = Temporary_Data.flatten()
            # Build a KDTree for efficient nearest neighbor search (on raster coordinates)
            Search_Tree = cKDTree(Temporary_Coordinates)
            # For each point in the GeoPackage, find the closest raster pixel
            GPKG_Coordinates   = np.array(list(zip(GeoDataFrames_Dict[Area].geometry.x, GeoDataFrames_Dict[Area].geometry.y)))
            Distances, Indices = Search_Tree.query(GPKG_Coordinates)     
        Nearest_Values     = Temporary_Values[Indices]
        # Compute processing time
        T2                = time.perf_counter()
        Processing_Time   = T2 - T1
        # Convert to h:m:s
        hours, rem       = divmod(Processing_Time, 3600)
        minutes, seconds = divmod(rem, 60)
        print(f"NN Task\t\t\t: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}") 

        # Start Auxiliary Counters
        Bicubic_Counter              = 0
        Bilinear_Counter             = 0
        Nearest_Counter              = 0
        Bicubic_Corrections_Counter  = 0
        Bilinear_Corrections_Counter = 0
        # Perform Bicubic Interpolation
        Bicubic_Values   = []
        i                = 0
        # Iterate Geopackage points and assign values to them
        for Coordinate_x, Coordinate_y in zip(GPKG_Coordinates_x, GPKG_Coordinates_y):
            if not(i%50000):
                print('Count:\t\t\t:' + str(i))
            # Convert to raster pixel coordinates
            Pixel_x, Pixel_y = gdal.ApplyGeoTransform(Temporary_INV_GT, Coordinate_x, Coordinate_y)
            Pixel_x_int, Pixel_y_int = int(Pixel_x), int(Pixel_y)
            # Verify if the closest value is No Data using the results from the Nearest Neighbour Interpolation
            if Nearest_Values[i] == Temporary_NoData or np.isnan(Nearest_Values[i]):
                Bicubic_Values.append(Temporary_NoData)
                i = i + 1
                continue
            # Verify if the Bicubic window (4x4) is within the raster boundaries
            x_start, y_start = Pixel_x_int - 1, Pixel_y_int - 1
            x_end, y_end     = x_start + 4, y_start + 4
            # Ensure the window is within raster boundaries
            if x_start < 0 or y_start < 0 or x_end > Temporary_Raster.RasterXSize or y_end > Temporary_Raster.RasterYSize:
                Bicubic_Values.append(Nearest_Values[i])
                i = i + 1
                continue
            # Get window for interpolation, bigger for the No Data
            Bicubic_Value, Bicubic_Corrections_Counter = bicubic_interpolation(Temporary_Data, Pixel_x, Pixel_y, Temporary_NoData, Bicubic_Corrections_Counter)
            # If No Data use the Nearest Neighbour Value
            if Bicubic_Value == Temporary_NoData or np.isnan(Bicubic_Value):
                # Try Bilinear Interpolation
                Bilinear_Value, Bilinear_Corrections_Counter = bilinear_interpolation(Temporary_Data, Pixel_x, Pixel_y, Temporary_NoData, Bilinear_Corrections_Counter)
                if Bilinear_Value == Temporary_NoData or np.isnan(Bilinear_Value):
                    Bicubic_Values.append(Nearest_Values[i])
                    Nearest_Counter = Nearest_Counter + 1
                else:
                    Bicubic_Values.append(Bilinear_Value)
                    Bilinear_Counter = Bilinear_Counter + 1
                i = i + 1
                continue
            Bicubic_Values.append(Bicubic_Value)
            Bicubic_Counter = Bicubic_Counter + 1
            i = i + 1
        # Compute processing time
        T3                = time.perf_counter()
        Processing_Time   = T3 - T2
        # Convert to h:m:s
        hours, rem       = divmod(Processing_Time, 3600)
        minutes, seconds = divmod(rem, 60)
        print(f"Interpolation time\t: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")   
        
        # Close the raster dataset
        Temporary_Raster = None
        # Add the interpolated data to the GeoPandas dataframe, under the biomass poll name
        Pool = Path.split('/')[-1].split('_')[2] + '_' + Path.split('/')[-1].split('_')[3]
        GeoDataFrames_Dict[Area][Pool] = Bicubic_Values
        if Path.split('/')[-1].split('_')[-2] in Biomass_Pools or Path.split('/')[-1].split('_')[-2] == 'PFT':
            GeoDataFrames_Dict[Area][Pool] = Nearest_Values
        # Compute processing time
        Time_Process_Ends = time.perf_counter()
        Processing_Time   = Time_Process_Ends - T0
        # Convert to h:m:s
        hours, rem       = divmod(Processing_Time, 3600)
        minutes, seconds = divmod(rem, 60)
        print(f"Pool time\t\t: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")   
        # Add to Biomass Pool if it was not added before
        print(Pool)
        print('-+-+-+-+-+-+-')

# Compute processing time
Time_Process_Ends = time.perf_counter()
Processing_Time   = Time_Process_Ends - Time_Process_Start
# Convert to h:m:s
hours, rem       = divmod(Processing_Time, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")

Mafra
Read Task		: 00:00:00.72
NN Task			: 00:00:05.73
Count:			:0
Interpolation time	: 00:00:00.08
Pool time		: 00:00:06.54
ALOS2_BETAHH
-+-+-+-+-+-+-
Read Task		: 00:00:00.76
NN Task			: 00:00:00.00
Count:			:0
Interpolation time	: 00:00:00.07
Pool time		: 00:00:00.84
ALOS2_BETAHV
-+-+-+-+-+-+-
Read Task		: 00:00:00.60
NN Task			: 00:00:00.00
Count:			:0
Interpolation time	: 00:00:00.07
Pool time		: 00:00:00.68
ALOS2_BETAND
-+-+-+-+-+-+-
Read Task		: 00:00:00.61
NN Task			: 00:00:00.00
Count:			:0
Interpolation time	: 00:00:00.08
Pool time		: 00:00:00.70
ALOS2_BETARATIO
-+-+-+-+-+-+-
Read Task		: 00:00:00.63
NN Task			: 00:00:00.00
Count:			:0
Interpolation time	: 00:00:00.11
Pool time		: 00:00:00.74
ALOS2_GAMMAHH
-+-+-+-+-+-+-
Read Task		: 00:00:00.62
NN Task			: 00:00:00.00
Count:			:0
Interpolation time	: 00:00:00.08
Pool time		: 00:00:00.71
ALOS2_GAMMAHV
-+-+-+-+-+-+-
Read Task		: 00:00:00.59
NN Task			: 00:00:00.00
Count:			:0
Interpolation time	: 00:00:00.08
Pool time		: 00:00:

c:\Users\guibe\Documents\Tese\.conda\Lib\site-packages\geopandas\geodataframe.py:1968: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)


Read Task		: 00:00:03.59
NN Task			: 00:00:00.00
Count:			:0
Interpolation time	: 00:00:00.08
Pool time		: 00:00:03.71
S1GRD_BETAVVVARIANCE
-+-+-+-+-+-+-
Read Task		: 00:00:03.61
NN Task			: 00:00:00.00
Count:			:0
Interpolation time	: 00:00:00.08
Pool time		: 00:00:03.73
S1GRD_BETAVV
-+-+-+-+-+-+-
Read Task		: 00:00:03.58
NN Task			: 00:00:00.00
Count:			:0
Interpolation time	: 00:00:00.08
Pool time		: 00:00:03.70
S1GRD_DEM
-+-+-+-+-+-+-
Read Task		: 00:00:03.64
NN Task			: 00:00:00.00
Count:			:0
Interpolation time	: 00:00:00.08
Pool time		: 00:00:03.77
S1GRD_GAMMAVHASM
-+-+-+-+-+-+-
Read Task		: 00:00:03.60
NN Task			: 00:00:00.00
Count:			:0
Interpolation time	: 00:00:00.08
Pool time		: 00:00:03.72
S1GRD_GAMMAVHCONTRAST
-+-+-+-+-+-+-
Read Task		: 00:00:03.58
NN Task			: 00:00:00.00
Count:			:0
Interpolation time	: 00:00:00.08
Pool time		: 00:00:03.71
S1GRD_GAMMAVHCORRELATION
-+-+-+-+-+-+-
Read Task		: 00:00:03.62
NN Task			: 00:00:00.00
Count:			:0
Interpolation time	: 00:00:00.08


In [8]:
for Key in GeoDataFrames_Dict.keys():
    GeoDataFrames_Dict[Key].to_file(Root_Directory + Key + '_' + BIOMASS_POOL + '_Dataset.gpkg', driver="GPKG")